# 🚀 Klint-32M: Generative Financial Foundation Architecture

> **Klint** is an open research foundation architecture for generative financial time-series modeling.
> Author: Akhilesh Varma (`akhverm@gmail.com`)  
> Target Scale: **Klint-32M** (~30.6M parameters: 10 layers, 480 hidden dimension, 10 heads).

This notebook provides the **complete, end-to-end production workflow**:
1. **Pre-training / Resuming** the 32M foundation model with zero-overhead In-VRAM tensor slicing.
2. **Generating** guaranteed structurally valid OHLCV synthetic trajectories ($H \ge \max(O, C), L \le \min(O, C)$).
3. **Rigorous Multi-Asset Testing Suite** testing across **300+ liquid assets across 6 asset classes** via `yfinance`.
4. **Automated Risk & Return Reporting**: Drawdown curves, Directional Hit Rate distributions, Sharpe rankings, and structured CSV logs.

*Note: All models, data, CSVs, and plots are stored directly inside the local environment (`./checkpoints/`, `./benchmarks/`) with zero Google Drive dependency.*

## 1. Hardware Detection & Accelerator Auto-Tuning
Detects available GPU (T4, V100, A100, or L4) and configures optimal precision.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU Detected: {gpu_name} ({total_vram:.2f} GB VRAM)")
    print(f"bfloat16 Acceleration Supported: {bf16_ok}")
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select GPU (T4 or A100).")

## 2. Environment Setup & Architectural Invariant Checks
Installs dependencies, ensures `klint` is in Python's path, and runs pytest to verify all invariant guarantees.

In [ ]:
import sys, os
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

# If running in Colab from cloned repo:
# !git clone https://github.com/ak495867/Klint-32M.git
# %cd /content/Klint-32M

!pip install -e ".[dev]"
!pip install yfinance matplotlib pandas scipy
!pytest tests/ -v
!mkdir -p checkpoints data benchmarks/reports benchmarks/plots

## 3. Stage 1: Factor Tokenizer Pre-training (RVQ Codebooks)
Pre-trains the 3 discrete factor codebooks (512 Price, 256 Range, 256 Activity) on market bars. (If you already have `checkpoints/tokenizer_best.pt`, this completes immediately).

In [ ]:
# Train or verify tokenizer codebooks
if not os.path.exists('checkpoints/tokenizer_best.pt'):
    print('Training tokenizer codebooks from scratch...')
    !python scripts/train_tokenizer.py \
        --data_path data/SOL.npy \
        --epochs 10 \
        --batch_size 128 \
        --learning_rate 0.001 \
        --save_dir checkpoints
else:
    print('Trained tokenizer found: checkpoints/tokenizer_best.pt (Ready)')

## 4. Stage 2: Discrete Token Caching (50x Training Acceleration)
Pre-encodes all 1.59M bars into integer token IDs `data/sol_tokens.pt`.

In [ ]:
if not os.path.exists('data/sol_tokens.pt'):
    !python scripts/cache_tokens.py \
        --data_path data/SOL.npy \
        --tokenizer_path checkpoints/tokenizer_best.pt \
        --output_path data/sol_tokens.pt \
        --batch_size 16384
else:
    print('Cached tokens found: data/sol_tokens.pt (Ready)')

## 5. Stage 3: High-Speed Klint-32M Foundation Model Training
**Resume & Turbocharged Training:** Loads existing weights (`checkpoints/klint_32m_step_1500.pt` or `checkpoints/klint_32m_best.pt`), puts tokens directly into GPU VRAM, and continues training forward at **30,000+ tokens/sec** with zero memory bus bottlenecks.

In [ ]:
# Prevent CUDA memory fragmentation
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# Check for existing checkpoint to resume from
resume_ckpt = None
for candidate in ['checkpoints/klint_32m_step_1500.pt', 'checkpoints/klint_32m_best.pt']:
    if os.path.exists(candidate):
        resume_ckpt = candidate
        break

resume_flag = f"--resume_from {resume_ckpt}" if resume_ckpt else ""
print(f"Training mode: {'RESUMING from ' + resume_ckpt if resume_ckpt else 'TRAINING FROM SCRATCH'}")

# Fast In-VRAM training: 256 bars context (768 tokens) = ~4.2h intraday dynamics
!python scripts/train_klint32m.py \
    --tokens_path data/sol_tokens.pt \
    --save_dir checkpoints \
    {resume_flag} \
    --max_steps 3000 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --learning_rate 0.0003 \
    --warmup_steps 200 \
    --eval_interval 200 \
    --save_interval 500 \
    --context_bars 256 \
    --no_gradient_checkpointing

## 6. Stage 4: Package Full Model Bundle for Easy Download
Combines the trained Tokenizer + Klint-32M weights + architecture configuration into `./checkpoints/klint_32m_release.pt`.

In [ ]:
import os, sys
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

import torch
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.tokenizer.factor_tokenizer import FactorTokenizer

bundle_path = "checkpoints/klint_32m_release.pt"
tok_ckpt = torch.load("checkpoints/tokenizer_best.pt", map_location="cpu", weights_only=False)
model_ckpt = torch.load("checkpoints/klint_32m_best.pt", map_location="cpu", weights_only=False)

torch.save({
    "architecture": "Klint-32M",
    "author": "Akhilesh Varma",
    "config": model_ckpt["config"],
    "tokenizer_state_dict": tok_ckpt,
    "model_state_dict": model_ckpt["model_state_dict"],
    "val_loss": model_ckpt.get("val_loss", None),
    "step": model_ckpt.get("step", None),
}, bundle_path)

bundle_size_mb = os.path.getsize(bundle_path) / (1024 * 1024)
print(f"Unified release bundle created: {bundle_path} ({bundle_size_mb:.2f} MB)")

# Optional: Trigger browser download to your local machine
# from google.colab import files
# files.download(bundle_path)

## 7. Stage 5: Generative Trajectory Sampling & Candlestick Visualization
Generates 100 synthetic market bars using causal factor routing ($P \to R \to A$) and verifies 100% geometric invariant satisfaction.

In [ ]:
import os, sys
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from klint.tokenizer.factor_tokenizer import FactorTokenizer
from klint.tokenizer.geometric_decoder import GeometricDecoder
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.data.validator import validate_ohlcv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torch.load("checkpoints/klint_32m_release.pt", map_location=device, weights_only=False)

tokenizer = FactorTokenizer().to(device)
tokenizer.load_state_dict(bundle["tokenizer_state_dict"])
tokenizer.eval()

model = Klint32M(bundle["config"]).to(device)
model.load_state_dict(bundle["model_state_dict"])
model.eval()
decoder = GeometricDecoder()

# Prompt with last 50 bars
tokens_data = torch.load("data/sol_tokens.pt", map_location="cpu", weights_only=False)
prompt_tokens = tokens_data["token_matrix"][-50:].view(1, -1).to(device)
num_future_bars = 100

gen_tokens = model.generate_tokens(prompt_tokens, num_bars=num_future_bars, temperature=0.8, top_k=40)
new_tokens = gen_tokens[:, 150:]

p_tok, r_tok, a_tok = tokenizer.deinterleave(new_tokens)
rec_p, rec_r, rec_a = tokenizer.decode_tokens(p_tok, r_tok, a_tok)
synth_bars = decoder(rec_p, rec_r, rec_a, anchor_price=float(tokens_data.get("anchor_price", 100.0))).squeeze(0).detach().cpu().numpy()

is_valid, report = validate_ohlcv(synth_bars)
print(f"Generated Candles Invariant Validity: {is_valid} | Violations: {report['total_violations']}")

# Candlestick Chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
for i in range(len(synth_bars)):
    o, h, l, c, v = synth_bars[i]
    color = '#26a69a' if c >= o else '#ef5350'
    ax1.plot([i, i], [l, h], color=color, linewidth=1.2)
    ax1.add_patch(plt.Rectangle((i - 0.35, min(o, c)), 0.7, max(abs(c - o), 0.001), color=color, alpha=0.9))
    ax2.bar(i, v, color=color, width=0.7, alpha=0.8)

ax1.set_title("Klint-32M: Synthetic Generated Market Trajectory (100 Bars)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Price ($)", fontsize=12)
ax1.grid(True, alpha=0.2)
ax2.set_ylabel("Volume", fontsize=12)
ax2.set_xlabel("Bar Step (t)", fontsize=12)
ax2.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("checkpoints/synthetic_market_trajectory.png", dpi=150)
plt.show()

## 8. Stage 6: Rigorous Multi-Asset Testing Suite (300+ Assets via yfinance)
Tests the trained Klint-32M foundation model across **300+ liquid assets across 6 asset classes** (Equities, ETFs, Crypto, Commodities, FX, Bonds):
* Calculates **Directional Hit Rate (%)**, **Annualized Sharpe**, **Sortino**, **Max Drawdown (%)**, and **Cumulative PnL**.
* Exports 4 comprehensive CSV scorecards to `benchmarks/reports/`.
* Generates and plots publication-quality performance graphs to `benchmarks/plots/`.

In [ ]:
# Execute the full multi-asset benchmark suite across 300+ assets
!python scripts/run_multi_asset_benchmark.py \
    --checkpoint checkpoints/klint_32m_best.pt \
    --tokenizer_checkpoint checkpoints/tokenizer_best.pt \
    --max_assets 320 \
    --period 1y \
    --interval 1d \
    --context_bars 64 \
    --output_dir benchmarks

## 9. Stage 7: Display Benchmark Visualizations & Asset Ranking
Displays the generated quantitative charts inline and previews the top-ranked assets.

In [ ]:
from IPython.display import Image, display
import pandas as pd

# 1. Display Visualizations
plots = [
    ("Equity & Underwater Drawdown Curves", "benchmarks/plots/drawdown_equity_curves.png"),
    ("Directional Accuracy Distribution vs 50% Random Walk", "benchmarks/plots/directional_accuracy_dist.png"),
    ("Top Asset Sharpe Ratio Rankings", "benchmarks/plots/asset_sharpe_ranking.png"),
    ("Cross-Asset Class Performance Comparison", "benchmarks/plots/asset_class_comparison.png"),
]

for title, path in plots:
    if os.path.exists(path):
        print(f"=== {title} ===")
        display(Image(filename=path))

# 2. Display Top 15 Ranked Assets Table
ranking_csv = "benchmarks/reports/asset_ranking.csv"
if os.path.exists(ranking_csv):
    df_rank = pd.read_csv(ranking_csv)
    cols = ["Rank", "Ticker", "Name", "Asset_Class", "Directional_Accuracy_Pct", "Annualized_Sharpe", "Max_Drawdown_Pct", "Cumulative_Return_Pct", "Win_Rate_Pct"]
    print("\n=== TOP 15 RANKED ASSETS BY ANNUALIZED SHARPE RATIO ===")
    display(df_rank[cols].head(15))

# 3. Display Asset Class Summary Table
class_csv = "benchmarks/reports/asset_class_aggregate.csv"
if os.path.exists(class_csv):
    df_class = pd.read_csv(class_csv)
    print("\n=== CROSS-ASSET CLASS AGGREGATE SUMMARY ===")
    display(df_class)